# 03. Entrenar modelos y generar predicciones

Este notebook ejecuta el ajuste supervisado local de T5S, BARTO, mBART-50 y uMT5-Base. Usa solamente los CSV congelados creados por el notebook 02. No emplea APIs de generación ni scripts externos.

**Importante:** `test.csv` no participa en el entrenamiento ni en la selección del checkpoint; solo se usa para generar las predicciones que evaluará el notebook 04.

## Requisitos

Con el entorno virtual activo, instala una vez las dependencias: `pip install -e '.[analysis,models,dev]'`. Los pesos base se descargarán desde Hugging Face la primera vez y se guardarán en su caché local.

## Variantes del experimento

El mismo conjunto de datos y los mismos modelos se entrenan por dos caminos:

| Variante | Qué cambia | Fuente en la carpeta de papers |
| --- | --- | --- |
| `epocas_5` | Ajuste completo, 5 épocas. Solo en `claims_500` reutiliza `claims_500_v1` | #99 (uMT5, 5 épocas) |
| `epocas_10` | Ajuste completo, 10 épocas | #100 (T5, 10 épocas) |
| `epocas_20` | Ajuste completo, 20 épocas | #80 (T5, 20 épocas) |
| `lora` | Solo se entrenan adaptadores LoRA: r=8, alpha=32, dropout=0,1; 10 épocas; learning rate 3e-4 | #75 (configuración), #80 (r=8), #100 (LoRA en T5) |

El learning rate de LoRA coincide con #97. En BARTO, con 3e-5 la pérdida de validación tras 2 épocas fue 2,26 y con 3e-4 fue 1,68. Las demás variantes conservan 3e-5.

Todas usan early stopping con paciencia 3 sobre la pérdida de validación (#66) y conservan el mejor checkpoint. Cambia `VARIANT` en la siguiente celda y ejecuta el notebook una vez por variante.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

DATASET_NAME = 'claims_1500'  # claims_500 sigue disponible para sus corridas previas
FINAL_DIR = ROOT / 'artifacts' / '02_conjuntos' / DATASET_NAME
from claim_extraction.workflow import DATASET_SPLIT_SIZES, EXPERIMENT_VARIANTS, run_name_for

# Elige una variante y vuelve a ejecutar el notebook: epocas_5, epocas_10, epocas_20 o lora.
VARIANT = 'epocas_10'
VARIANT_CONFIG = EXPERIMENT_VARIANTS[VARIANT]
EARLY_STOPPING_PATIENCE = 3  # épocas sin mejora en validación antes de detener (paper 66)
RUN_NAME = run_name_for(DATASET_NAME, VARIANT)  # cada conjunto y variante tiene su carpeta
RUN_DIR = ROOT / 'artifacts' / '03_corridas' / RUN_NAME
SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
EXPECTED_SPLIT_SIZES = DATASET_SPLIT_SIZES[DATASET_NAME]
# Parámetros aproximados del checkpoint completo; este flujo ajusta todos sus pesos.
MODEL_IDS = [
    't5s_spanish',  # T5S: ≈248 M parámetros.
    'barto_spanish',  # BARTO: ≈140 M parámetros.
    'mbart50_es',  # mBART-50: ≈610 M parámetros.
    'umt5_base_es',  # uMT5-Base: ≈590 M parámetros.
]
MODELS_TO_RUN = MODEL_IDS  # reduce esta lista para ejecutar solo los modelos pendientes

# Hiperparámetros comunes: sus valores actuales se definen en configs/models.json.
# epochs: los fija la variante elegida en VARIANT, con early stopping sobre validación.
# learning_rate=3e-5: tamaño de cada actualización de pesos.
# weight_decay=0: penalización de pesos grandes; 0 la desactiva.
# warmup_steps=0: aumento gradual inicial del learning rate; 0 lo desactiva.
# effective_batch_size=8: ejemplos por actualización; con BATCH_SIZE=1 acumula 8 pasos.
# max_source_length=512: máximo de tokens de título+cuerpo; el excedente se trunca.
# max_target_length=96: máximo de tokens del claim de referencia durante entrenamiento.
# num_beams=4: alternativas que compara beam search; más beams suele costar más tiempo.
# max_new_tokens=96: límite de tokens al generar el claim.
BATCH_SIZE = 1  # microbatch que cabe en memoria; no cambia el batch efectivo configurado.
SEED = 20261103  # valor fijo para repetir la aleatoriedad de esta corrida
PROGRESS_EVERY_PERCENT = 5  # usa 1 si quieres ver cada punto porcentual
CACHE_DIR = None  # por ejemplo: Path('/ruta/a/cache_huggingface')
RESUME = True  # True permite reutilizar checkpoints y predicciones ya completos

# Bloque uMT5 en MPS: reduce memoria al recalcular activaciones durante la retropropagación.
# No cambia los datos, épocas, batch efectivo ni longitudes; sí puede tardar más por esa recomputación.
GRADIENT_CHECKPOINTING_MODELS = {'umt5_base_es'}

print({'variant': VARIANT, 'dataset': DATASET_NAME, 'final_dir': str(FINAL_DIR), 'run_dir': str(RUN_DIR), 'models': MODELS_TO_RUN})

In [ ]:
import json
import platform

from claim_extraction.registry import get_model
from claim_extraction.workflow import (
    available_device,
    generate_predictions,
    train_seq2seq_model,
    utc_now,
    split_fingerprints,
    validate_sem1_splits,
)

def show_training_progress(model_id, percent, current_step, total_steps):
    print(f'{model_id}: {percent}% ({current_step}/{total_steps} pasos)', flush=True)

if not set(MODELS_TO_RUN).issubset(MODEL_IDS):
    raise ValueError('MODELS_TO_RUN debe contener IDs definidos en MODEL_IDS.')
if len(MODELS_TO_RUN) != len(set(MODELS_TO_RUN)):
    raise ValueError('Cada modelo debe aparecer una sola vez.')
if BATCH_SIZE < 1:
    raise ValueError('BATCH_SIZE debe ser positivo.')
if not 1 <= PROGRESS_EVERY_PERCENT <= 100:
    raise ValueError('PROGRESS_EVERY_PERCENT debe estar entre 1 y 100.')

splits = validate_sem1_splits(SPLIT_PATHS, expected_sizes=EXPECTED_SPLIT_SIZES)
SPLIT_FINGERPRINTS = split_fingerprints(splits)
print({name: len(frame) for name, frame in splits.items()})
print({'device': available_device(), 'seed': SEED, 'progress_every_percent': PROGRESS_EVERY_PERCENT})

## Crear o retomar la corrida

Una carpeta de corrida conserva checkpoints, predicciones y un manifiesto local. Con `RESUME = False`, el notebook se niega a mezclar una corrida anterior con una nueva.

In [ ]:
from claim_extraction.workflow import assert_run_matches

# El manifiesto deja registrados los parámetros de la corrida y evita mezclar configuraciones.
RUN_MANIFEST = RUN_DIR / 'run_manifest.json'
STATUS_PATH = RUN_DIR / 'run_status.json'

if RUN_DIR.exists() and any(RUN_DIR.iterdir()) and not RESUME:
    raise FileExistsError(
        f'La corrida ya existe en {RUN_DIR}. Activa RESUME o cambia RUN_NAME; no se sobrescribe.'
    )

for directory in (RUN_DIR / 'models', RUN_DIR / 'predictions'):
    directory.mkdir(parents=True, exist_ok=True)

if RUN_MANIFEST.is_file():
    run_manifest = json.loads(RUN_MANIFEST.read_text(encoding='utf-8'))
    # RESUME reutiliza checkpoints: deben venir de estos mismos datos y variante.
    assert_run_matches(run_manifest, dataset_name=DATASET_NAME, variant=VARIANT, fingerprints=SPLIT_FINGERPRINTS)
else:
    run_manifest = {
        'dataset_name': DATASET_NAME,
        'run_name': RUN_NAME,
        'started_at_utc': utc_now(),
        'device': available_device(),
        'platform': platform.platform(),
        'seed': SEED,
        'variant': VARIANT,
        'variant_config': VARIANT_CONFIG,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'split_rows': {name: len(frame) for name, frame in splits.items()},
        'split_fingerprints': SPLIT_FINGERPRINTS,
        'models': {model_id: {
            'display_name': get_model(model_id).display_name,
            'base_checkpoint': get_model(model_id).base_checkpoint,
            'training': get_model(model_id).training,
            'generation': get_model(model_id).generation,
        } for model_id in MODELS_TO_RUN},
    }
    RUN_MANIFEST.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

status = {'status': 'running', 'updated_at_utc': utc_now(), 'models': {}}
STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Corrida preparada: {RUN_DIR.relative_to(ROOT)}')

## Ejecutar entrenamiento e inferencia

Esta celda puede tardar. Cada modelo se ajusta con `train.csv`, selecciona su checkpoint por pérdida de validación y luego genera un CSV de predicciones sobre los casos de prueba. Si una ejecución se interrumpe, activa `RESUME` para reutilizar los componentes ya terminados.

In [ ]:
# Todos los modelos se entrenan con el mismo train/validation y predicen el mismo test fijo.
for model_id in MODELS_TO_RUN:
    model_dir = RUN_DIR / 'models' / model_id
    checkpoint = model_dir / 'best'
    prediction_path = RUN_DIR / 'predictions' / f'{model_id}.csv'
    try:
        if checkpoint.is_dir() and RESUME:
            print(f'{model_id}: checkpoint existente; se reutiliza.')
        else:
            print(f'{model_id}: entrenamiento iniciado.')
            status['models'][model_id] = {'status': 'training'}
            STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
            manifest = train_seq2seq_model(
                model_id=model_id,
                train=splits['train'],
                validation=splits['validation'],
                output_dir=model_dir,
                seed=SEED,
                batch_size=BATCH_SIZE,
                cache_dir=CACHE_DIR,
                progress_reporter=lambda percent, current_step, total_steps, current_model=model_id: show_training_progress(current_model, percent, current_step, total_steps),
                progress_every_percent=PROGRESS_EVERY_PERCENT,
                gradient_checkpointing=model_id in GRADIENT_CHECKPOINTING_MODELS,
                epochs=VARIANT_CONFIG['epochs'],
                learning_rate=VARIANT_CONFIG.get('learning_rate'),
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                lora=VARIANT_CONFIG['lora'],
            )
            print(f"{model_id}: entrenamiento terminado en {manifest['device']}.")

        if prediction_path.is_file() and RESUME:
            print(f'{model_id}: predicciones existentes; se reutilizan.')
        else:
            status['models'][model_id] = {'status': 'generating', 'checkpoint': str(checkpoint)}
            STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
            generated = generate_predictions(
                model_id=model_id,
                checkpoint=checkpoint,
                candidates=splits['test'],
                output_path=prediction_path,
                batch_size=BATCH_SIZE,
                cache_dir=CACHE_DIR,
            )
            if len(generated) != len(splits['test']):
                raise AssertionError(f"{model_id} no generó las {len(splits['test'])} predicciones esperadas.")

        status['models'][model_id] = {
            'status': 'ready_for_evaluation',
            'checkpoint': str(checkpoint),
            'predictions': str(prediction_path),
        }
        status['updated_at_utc'] = utc_now()
        STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as exc:
        status['models'][model_id] = {'status': 'failed', 'error': str(exc)}
        status['status'] = 'failed'
        status['updated_at_utc'] = utc_now()
        STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
        raise

status['status'] = 'ready_for_evaluation'
status['updated_at_utc'] = utc_now()
STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
print('Listo. Abre el notebook 04 para calcular las métricas sobre test.csv.')

In [ ]:
import pandas as pd

pd.DataFrame([
    {'modelo': model_id, **details}
    for model_id, details in status['models'].items()
])